# Introdução ao Projeto: Análise Exploratória de Dados

## Contexto  
Este projeto tem como base um dataset de previsão de vendas disponível no Kaggle https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting. O conjunto de dados reúne informações detalhadas sobre transações, desempenho financeiro e outros indicadores comerciais, permitindo uma análise aprofundada dos padrões de vendas e das variações do mercado.

## Problema  
Os dados revelam variações significativas entre diferentes categorias, regiões e períodos. Essa variedade torna desafiador identificar os fatores-chave que impulsionam o desempenho e a rentabilidade, exigindo uma análise criteriosa para extrair insights relevantes.

## Objetivo  
O objetivo principal deste projeto é conduzir uma Análise Exploratória de Dados (EDA) completa, que permita:
- Compreender a dinâmica e distribuição das vendas;
- Identificar tendências e padrões relevantes;
- Reconhecer oportunidades de melhoria e áreas com potencial de crescimento.

## Etapas do Projeto  
- **Importação e Compreensão dos Dados:** Carregar o dataset e realizar uma análise inicial para entender a estrutura, dimensões e principais variáveis.  
- **Limpeza e Tratamento dos Dados:** Detectar e corrigir inconsistências, além de tratar valores ausentes para garantir a confiabilidade da análise.  
- **Análise Estatística Descritiva:** Aplicar técnicas de resumo estatístico para captar as características essenciais dos dados.  
- **Visualizações Interativas:** Desenvolver gráficos e dashboards para uma exploração visual eficaz dos padrões e tendências.  
- **Geração de Insights e Recomendações:** Traduzir os achados da análise em insights que possam gerar decisões estratégicas.


##  Configuração do Ambiente

### Instalação das Bibliotecas

In [ ]:
%pip install pandas numpy matplotlib seaborn plotly

### Importação das Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## Carregamento e Inspeção dos Dados

### Carregamento do Dataset

In [ ]:
caminho = './data/train.csv'

df = pd.read_csv("./data/train.csv")
df_original = df.copy() 

print(f"Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")
df.head()

### Examinando Informações do Dataset

In [ ]:
df.info()

### Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas para variáveis numéricas
df.describe()

In [ ]:
# Estatísticas descritivas para variáveis categóricas
df.describe(include='object')

### Identificação de Valores Ausentes
Verificamos a quantidade e a proporção de valores ausentes em cada coluna.

In [ ]:
# Calculando valores ausentes
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

# Criando um dataframe para visualizar os valores ausentes
missing_df = pd.DataFrame({
    'Valores Ausentes': missing_values,
    'Porcentagem (%)': missing_percent
})

# Exibindo apenas colunas com valores ausentes
missing_df = missing_df[missing_df['Valores Ausentes'] > 0].sort_values('Porcentagem (%)', ascending=False)
missing_df

In [ ]:
# Contar valores ausentes por coluna
missing_count = df.isnull().sum()
print(missing_count[missing_count > 0])

In [ ]:
# Analisando os registros com valores ausentes na coluna 'Postal Code' para verificar padrões
df[df['Postal Code'].isna()]


Identificamos que todos os registros com valores NaN na coluna 'Postal Code' correspondem exclusivamente ao estado de Vermont na região Leste.

### Tratamento de Valores Ausentes
Preenchemos os valores ausentes com o código 5001, que corresponde a um código postal comum utilizado em Burlington, Vermont. Esta abordagem foi escolhida por ser a principal cidade do estado, garantindo consistência nos dados.

In [ ]:
# Verificando padrões nos códigos postais
print(f"Valores únicos em Postal Code: {df['Postal Code'].nunique()}")

# preenchendo valores ausentes com o código postal de vermont
df['Postal Code'] = df['Postal Code'].fillna(5001)
print(f"Preenchidos valores ausentes com o Postal Code de Vermont: {5001}")

# Verificando se ainda existem valores ausentes
print(f"Valores ausentes restantes: {df.isnull().sum().sum()}")

### Verificação de linhas duplicadas

In [ ]:
duplicates = df.duplicated()
total_duplicates = duplicates.sum()
print(f"Total de linhas duplicadas no dataset: {total_duplicates}")

## Análise de Variáveis Categóricas

### Identificação de Variáveis Categóricas

In [ ]:
# Identificando variáveis categóricas
cat_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"Variáveis categóricas: {cat_columns}")

### Distribuição de Categorias

In [ ]:


def plot_categorical_distribution(ax, df, column_name, palette_name="Set2"):
    counts = df[column_name].value_counts().sort_values(ascending=False).reset_index()
    counts.columns = [column_name, 'count']  

    palette = sns.color_palette(palette_name, n_colors=len(counts))

    sns.barplot(data=counts, x=column_name, y='count', hue=column_name, 
                ax=ax, palette=palette, legend=False)
    
    ax.set_title(f'Distribuição de {column_name}', fontsize=12)
    ax.set_ylabel('Contagem')
    ax.set_xlabel(column_name)
    ax.tick_params(axis='x')

# Criando o grid 2x2
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Lista de variáveis categóricas e seus respectivos eixos
variaveis = ['Ship Mode', 'Segment', 'Category', 'Region']
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

# Gerando os gráficos com a função
for var, pos in zip(variaveis, positions):
    plot_categorical_distribution(axes[pos], df, var)

plt.tight_layout()
plt.show()


### Análise da Relação entre Variáveis Categóricas

In [ ]:
# Criando uma tabela para analizar a relação entre Segment e Category
segment_category = pd.crosstab(df['Segment'], df['Category'])
print("Tabela cruzada: Segment x Category")
segment_category

In [ ]:
# Visualizando a relação entre Segment e Category
fig, ax = plt.subplots(figsize=(12, 9))  # Cria figura E eixos
segment_category.plot(kind='bar', colormap='viridis', ax=ax) 
plt.title('Distribuição de Categorias por Segmento', fontsize=14)
plt.xlabel('Segmento')
plt.ylabel('Contagem')
plt.legend(title='Categoria')
plt.tight_layout()
plt.show()

## Análise de Variáveis Numéricas

In [ ]:
# Identificando variáveis numéricas
num_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Variáveis numéricas: {num_columns}")

### Distribuição de Variáveis Numéricas

In [ ]:
## Verificando a distribuição de vendas e possíveis outliers através de um boxplot
plt.figure(figsize=(12, 9))
sns.boxplot(x=df['Sales'], color='skyblue')
plt.title('Boxplot de Sales')
plt.xlabel('Valor das Vendas')

# Anotação com as principais estatísticas 
stats = df['Sales'].describe()
info = f"Mediana: {stats['50%']:.2f} | Q1: {stats['25%']:.2f} | Q3: {stats['75%']:.2f} | Máx: {stats['max']:.2f}"
plt.annotate(info, xy=(0.5, 0.95), xycoords='axes fraction', ha='center',
             bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.9))

plt.tight_layout()
plt.show()

### Análise Bivariada

In [ ]:
# Boxplots para vendas por região
plt.figure(figsize=(12, 9))
sns.boxenplot(x='Region', hue='Region', y='Sales', data=df, palette='deep', legend=False)
plt.title('Distribuição de Vendas por Região', fontsize=14)
plt.xlabel('Região')
plt.ylabel('Vendas')
plt.grid(True, alpha=0.9)
plt.show()

 ## Análise Temporal

In [ ]:
# Convertendo colunas de data para o tipo datetime para facilitar a análise temporal
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

# Extraindo componentes das datas
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Day'] = df['Order Date'].dt.day
df['Order Weekday'] = df['Order Date'].dt.day_name()
df['Ship Year'] = df['Ship Date'].dt.year
df['Ship Month'] = df['Ship Date'].dt.month

# Calculando o tempo de entrega (em dias)
df['Delivery Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# Visualizando as novas colunas
df[['Order Date', 'Ship Date', 'Order Year', 'Order Month', 'Order Weekday', 'Delivery Time']].head()

### Análise de Tendências Temporais

In [ ]:
# Agrupando vendas por ano e mês
vendas_temporais = df.groupby(['Order Year', 'Order Month']).agg({
    'Sales': 'sum',
    'Order ID': 'count'
}).reset_index()
vendas_temporais.rename(columns={'Order ID': 'Pedidos'}, inplace=True)

# Criando uma data combinada para facilitar a visualização
vendas_temporais['Data'] = pd.to_datetime(vendas_temporais['Order Year'].astype(str) + '-' + 
                                         vendas_temporais['Order Month'].astype(str) + '-01')
vendas_temporais.sort_values('Data', inplace=True)

# Visualizando vendas ao longo do tempo
plt.figure(figsize=(16, 8))
sns.lineplot(x='Data', y='Sales', data=vendas_temporais, marker='o', linewidth=2)
plt.title('Evolução de Vendas ao Longo do Tempo', fontsize=14)
plt.xlabel('Data')
plt.ylabel('Vendas Totais')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Agrupando as vendas por mês para identificar padrões sazonais
sazonalidade_mensal = df.groupby('Order Month').agg({
    'Sales': 'sum',
    'Order ID': 'count'
}).reset_index()
sazonalidade_mensal.rename(columns={'Order ID': 'Pedidos'}, inplace=True)

# Criando nomes dos meses para o eixo X
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
sazonalidade_mensal['Mês'] = sazonalidade_mensal['Order Month'].apply(lambda x: meses[x-1])

# Visualizando sazonalidade mensal
plt.figure(figsize=(14, 7))
sns.barplot(x='Mês',hue='Mês', y='Sales', data=sazonalidade_mensal, palette='viridis', legend=False)
plt.title('Sazonalidade Mensal de Vendas', fontsize=14)
plt.xlabel('Mês')
plt.ylabel('Vendas Totais')
plt.grid(True, alpha=0.3)
plt.show()

### Conclusões

Após a análise exploratória detalhada, podemos destacar os seguintes insights:

1. **Perfil de Vendas:**
   - A maioria dos pedidos utiliza o modo de envio "Standard Class".
   - O segmento "Consumer" representa a maior parte das vendas.
   - A categoria com maior número de vendas é "Office supplies".

2. **Análise Regional:**
   - A região "West" apresenta os maiores valores de vendas.

3. **Padrões Temporais:**
   - Identificamos sazonalidade nas vendas, com picos nos meses de Setembro, Novembro e Dezembro.

